In [1]:
# Packages
import os
import re
import time
import glob
from pathlib import Path
from functools import partial
from multiprocessing import Pool, cpu_count
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.features import shapes
from scipy import ndimage
from scipy.ndimage import generic_filter
import dask.array as da

In [2]:
print("Current directory:", os.getcwd())
print("Files:", os.listdir())
files = glob.glob("/home/gisuser/code/data/DS_TEST/*")
print(files)

Current directory: /home/gisuser/code
Files: ['.git', '.gitignore', 'code', 'data', 'Dockerfile', 'environment.yml', 'mann_kendall_tau_result.tif', 'MSPA_2000_P.tif', 'MSPA_2000_P.tif.aux.xml', 'MSPA_2005_P.tif', 'MSPA_2005_P.tif.aux.xml', 'notes', 'outputs', 'README.md']
['/home/gisuser/code/data/DS_TEST/1991_P_recoded_mspa', '/home/gisuser/code/data/DS_TEST/extent_weird', '/home/gisuser/code/data/DS_TEST/mosaic', '/home/gisuser/code/data/DS_TEST/mosaic_P_recoded', '/home/gisuser/code/data/DS_TEST/MSPA', '/home/gisuser/code/data/DS_TEST/MSPA_clip', '/home/gisuser/code/data/DS_TEST/MSPA_rc_edge', '/home/gisuser/code/data/DS_TEST/MSPA_renamed', '/home/gisuser/code/data/DS_TEST/temp_hold_for_code', '/home/gisuser/code/data/DS_TEST/TRASH']


In [2]:
#######################################################
# Parameters - Set these variables before running
#######################################################

# Rename MSPA output
mspa_in = "/home/gisuser/code/data/DS_TEST/MSPA"
mspa_renamed = "/home/gisuser/code/data/DS_TEST/MSPA_renamed"

# Clip parameters (optional)
clip_in = "/home/gisuser/code/data/DS_TEST/MSPA_renamed"#r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\clip_in"
clip_mask = "/home/gisuser/code/data/DS_TEST/temp_hold_for_code/_00_05_combined_rc_P_tif_c_tif_nsf2.tif"#r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\binary_mask\binary_mask_shrink40.tif"
clip_shapefile = None #r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\boundary.shp"
clip_out = "/home/gisuser/code/data/DS_TEST/MSPA_clip"#r"S:\Mikayla\DATA\Projects\AF\NEW_WORKING\clip_out"

# Reclassification
rc_in = "/home/gisuser/code/data/DS_TEST/MSPA_clip"#r"D:\typology\data\DS_TEST\MSPA_clip" #r"D:\typology\data\DS_TEST\MSPA_renamed"
edge_rc_out = "/home/gisuser/code/data/DS_TEST/MSPA_rc_edge"#r"D:\typology\data\DS_TEST\MSPA_rc_edge"
area_rc_out = "/home/gisuser/code/data/DS_TEST/MSPA_rc_area"#r"D:\typology\data\DS_TEST\MSPA_rc_area"
rc_type = "edge"  # "edge" or "area"

## Patch Number Calculation 
# Region group
rg_out = r"D:\typology\data\DS_TEST\rg"
neighbor = "EIGHT"  # "FOUR" or "EIGHT"
# Reclass region group
rc_rg_in = r"D:\typology\data\DS_TEST\rg"
rc_rg_out = r"D:\typology\data\DS_TEST\rg_rc"

# Moving window
edge_mw_in = "/home/gisuser/code/data/DS_TEST/MSPA_rc_edge"
#area_mw_in = r"D:\typology\data\DS_TEST\MSPA_rc_area"
#pn_mw_in = r"D:\typology\data\DS_TEST\rg_rc"
mw_out = "/home/gisuser/code/data/DS_TEST/mw_results" #r"D:\typology\data\DS_TEST\mw_results"
mw_type = "edge"  # "edge", "area", or "pn"
mw_radius = 1000  # in map units (meters)
stat = "SUM"  # "SUM" or "VARIETY"

# Processing
n_workers = cpu_count()
chunk_size = 2048


In [4]:
#######################################################
# UTILITY FUNCTIONS
#######################################################

# rename MSPA output files
def rename_mspa_files(input_dir, output_dir):
    """
    Rename MSPA files from YEAR_P_recoded_8_1_0_1.tif to MSPA_YEAR_P.tif
    
    Args:
        input_dir: Directory with MSPA output files
        output_dir: Directory for renamed files
    """
    try:
        input_dir = Path(input_dir)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        files = list(input_dir.glob("*_P_recoded_*.tif"))
        
        for file in files:
            year = get_year(file.name)
            new_name = f"MSPA_{year}.tif"
            output_path = output_dir / new_name
            
            # Copy file with new name
            import shutil
            shutil.copy2(file, output_path)
            print(f"Renamed: {file.name} -> {new_name}")
        
        print(f"Renamed {len(files)} files")
        
    except Exception as e:
        print(f"Error renaming files: {e}")


def get_year(filename):
    """Extract 4-digit year from filename"""
    match = re.search(r"(\d{4})", filename)
    return match.group(1) if match else ""


def get_raster_files(directory, pattern="*.tif"):
    """Get list of raster files in directory"""
    rasters = sorted(Path(directory).glob(pattern))
    
    if not rasters:
        print(f"No rasters found in directory: {directory}")
        return []
    
    print(f"Found {len(rasters)} rasters")
    return rasters


# Batch Processing
def process_rasters_parallel(process_func, input_files, n_workers=n_workers, **kwargs):
    """Process multiple rasters in parallel"""
    if not input_files:
        return []
    
    print(f"Processing with {n_workers} workers...")
    
    func = partial(process_func, **kwargs)
    
    with Pool(processes=n_workers) as pool:
        outputs = pool.map(func, input_files)
    
    success_count = sum(1 for p in outputs if p)
    print(f"Process complete: {success_count}/{len(input_files)} succeeded")
    
    return outputs

#######################################################
# PROCESSING FUNCTIONS
#######################################################

def clip_raster(input_path, output_dir, bbox=None, mask_path=None):
    """
    Clip raster using bounding box or mask raster
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        bbox: Tuple of (minx, miny, maxx, maxy) in raster CRS, or None
        mask_path: Path to mask raster, or None
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        year = get_year(input_path.name)
        output_path = output_dir / f"{year}_clipped.tif"
        
        print(f"Clipping {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            if bbox:
                # Clip using bounding box
                from rasterio.windows import from_bounds
                window = from_bounds(*bbox, transform=src.transform)
                clipped = src.read(window=window)
                transform = src.window_transform(window)
                
            elif mask_path:
                # Clip using mask raster
                with rasterio.open(mask_path) as mask_src:
                    mask_data = mask_src.read(1)
                    mask_geom = [{'type': 'Polygon', 'coordinates': [list(shape['coordinates'][0])]} 
                                 for shape, value in shapes(mask_data, transform=mask_src.transform) 
                                 if value == 1]
                clipped, transform = mask(src, mask_geom, crop=True)
            else:
                print("Error: Must provide either bbox or mask_path")
                return None
            
            kwargs = src.meta.copy()
            kwargs.update({
                'height': clipped.shape[1],
                'width': clipped.shape[2],
                'transform': transform,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(clipped)
        
        print(f"Successfully clipped: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error clipping {input_path}: {e}")
        return None


def reclassify_mspa(input_path, rc_type, edge_out_dir, area_out_dir):
    """
    Reclassify MSPA output: edge (3, 103, 105) or area (3, 103, 105, 17, 117) to 1, rest to 0
    
    Args:
        input_path: Path to input raster
        rc_type: "edge" or "area"
        edge_out_dir: Output directory for edge
        area_out_dir: Output directory for area
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        year = get_year(input_path.name)
        
        if rc_type == "edge":
            output_dir = Path(edge_out_dir)
            output_path = output_dir / f"{year}_rc_edge.tif"
            target_values = [3, 103, 105]
        elif rc_type == "area":
            output_dir = Path(area_out_dir)
            output_path = output_dir / f"{year}_rc_area.tif"
            target_values = [3, 103, 105, 17, 117]
        else:
            print(f"Error: Undefined rc_type '{rc_type}'. Must be 'edge' or 'area'.")
            return None
        
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Reclassifying {input_path.name} ({rc_type})...")
        
        with rasterio.open(input_path) as src:
            data = da.from_array(src.read(1), chunks=(chunk_size, chunk_size))
            
            # Create mask for target values
            mask = da.zeros_like(data, dtype=bool)
            for val in target_values:
                mask = mask | (data == val)
            
            # Explicit: target values = 1, everything else = 0
            result = da.where(mask, 1, 0).astype(np.uint8)
            
            result_computed = result.compute()
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'uint8',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result_computed, 1)
        
        print(f"Successfully reclassified: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reclassifying {input_path}: {e}")
        return None


def region_group(input_path, output_dir, connectivity=8):
    """
    Apply region grouping (connected component labeling) to identify patches
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        connectivity: 4 or 8 neighbor connectivity
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        year = get_year(input_path.name)
        output_path = output_dir / f"{year}_area_rg.tif"
        
        print(f"Region grouping {input_path.name}...")
        
        # Map connectivity
        struct = ndimage.generate_binary_structure(2, connectivity // 4)
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            
            # Label connected components (exclude 0)
            binary = data > 0
            labeled, num_features = ndimage.label(binary, structure=struct)
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'int32',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(labeled.astype(np.int32), 1)
        
        print(f"Successfully region grouped: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error region grouping {input_path}: {e}")
        return None


def reclassify_rg(input_path, output_dir):
    """
    Reclassify region group raster: set value 1 to 0, keep rest
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        output_path = output_dir / f"{input_path.stem}_rc.tif"
        
        print(f"Reclassifying RG {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            result = np.where(data == 1, 0, data)
            
            kwargs = src.meta.copy()
            kwargs.update({'compress': 'lzw'})
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result, 1)
        
        print(f"Successfully reclassified RG: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error reclassifying RG {input_path}: {e}")
        return None


def moving_window(input_path, output_dir, mw_type, radius, stat):
    """
    Apply moving window analysis with specified radius and statistic
    
    Args:
        input_path: Path to input raster
        output_dir: Output directory
        mw_type: Type identifier ("edge", "area", "pn")
        radius: Radius in map units
        stat: "SUM" or "VARIETY"
    
    Returns:
        Path to output raster if successful, None otherwise
    """
    try:
        input_path = Path(input_path)
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        
        year = get_year(input_path.name)
        output_path = output_dir / f"{year}_{mw_type}_1km.tif"
        
        print(f"Moving window on {input_path.name}...")
        
        with rasterio.open(input_path) as src:
            data = src.read(1)
            pixel_size = src.transform[0]
            
            # Calculate radius in pixels
            radius_pixels = int(radius / pixel_size)
            
            # Create circular kernel
            y, x = np.ogrid[-radius_pixels:radius_pixels+1, -radius_pixels:radius_pixels+1]
            kernel = x**2 + y**2 <= radius_pixels**2
            
            # Apply focal statistic
            if stat == "SUM":
                result = ndimage.generic_filter(data.astype(np.float32), np.sum, footprint=kernel, mode='constant', cval=0)
            elif stat == "VARIETY":
                def variety(values):
                    return len(np.unique(values[values > 0]))
                result = ndimage.generic_filter(data, variety, footprint=kernel, mode='constant', cval=0)
            else:
                print(f"Error: Unsupported statistic '{stat}'")
                return None
            
            kwargs = src.meta.copy()
            kwargs.update({
                'dtype': 'float32',
                'nodata': None,
                'compress': 'lzw'
            })
            
            with rasterio.open(output_path, 'w', **kwargs) as dst:
                dst.write(result.astype(np.float32), 1)
        
        print(f"Successfully processed moving window: {output_path.name}")
        return str(output_path)
    
    except Exception as e:
        print(f"Error in moving window {input_path}: {e}")
        return None


In [5]:
# #######################################################
# # Main 
# #######################################################

if __name__ == "__main__":
    print("Start Processing")
    
    # # Rename MSPA output
    # print("\nStarting Rename MSPA Output")
    # rename_mspa_files(mspa_in, mspa_renamed)
    
    # # Clip- if applicable choose method by setting one to None
    # print("\nStarting Clip")
    # clip_start = time.time()
    # input_files = get_raster_files(clip_in)

    # if clip_mask:
    #     # Option 1: Clip with mask raster
    #     clip_results = process_rasters_parallel(
    #         clip_raster,
    #         input_files,
    #         output_dir=clip_out,
    #         mask_path=clip_mask
    #     )
    # elif clip_shapefile:
    #     # Option 2: Clip with shapefile
    #     clip_results = process_rasters_parallel(
    #         clip_raster,
    #         input_files,
    #         output_dir=clip_out,
    #         shapefile_path=clip_shapefile
    #     )
    # else:
    #     print("No clip method specified")

    # print(f"Clip completed in {time.time() - clip_start:.2f} seconds")
      
    # # Reclassify
    # print("\nStarting Reclassification")
    # rc_start = time.time()
    # input_files = get_raster_files(rc_in)
    # if input_files:
    #     rc_results = process_rasters_parallel(
    #         reclassify_mspa,
    #         input_files,
    #         rc_type=rc_type,
    #         edge_out_dir=edge_rc_out,
    #         area_out_dir=area_rc_out
    #     )
    # print(f"Reclassification completed in {time.time() - rc_start:.2f} seconds")
    
#     # # Region Group- for Patch Number only
#     # print("\nStarting RegionGroup")
#     # rg_start = time.time()
#     # input_files = get_raster_files(area_rc_out)
#     # if input_files:
#     #     rg_results = process_rasters_parallel(
#     #         region_group,
#     #         input_files,
#     #         output_dir=rg_out,
#     #         connectivity=8 if neighbor == "EIGHT" else 4
#     #     )
#     # print(f"RegionGroup completed in {time.time() - rg_start:.2f} seconds")
    
#     # # Reclass Region Group- for Patch Number only
#     # print("\nStarting Reclass Region Group")
#     # rc_rg_start = time.time()
#     # input_files = get_raster_files(rc_rg_in)
#     # if input_files:
#     #     rc_rg_results = process_rasters_parallel(
#     #         reclassify_rg,
#     #         input_files,
#     #         output_dir=rc_rg_out
#     #     )
#     # print(f"Reclass Region Group completed in {time.time() - rc_rg_start:.2f} seconds")
    
    ## Moving Window
    print("\nStarting Moving Window")
    mw_start = time.time()
    input_files = get_raster_files(edge_mw_in)  # Change based on mw_type
    if input_files:
        mw_results = process_rasters_parallel(
            moving_window,
            input_files,
            output_dir=mw_out,
            mw_type=mw_type,
            radius=mw_radius,
            stat=stat
        )
    print(f"Moving window completed in {time.time() - mw_start:.2f} seconds")
    
    print("\nProcessing complete!")


Start Processing

Starting Moving Window
Found 1 rasters
Processing with 16 workers...


Moving window on 1991_rc_edge.tif...
Successfully processed moving window: 1991_edge_1km.tif
Process complete: 1/1 succeeded
Moving window completed in 4746.67 seconds

Processing complete!
